# Introduction to QDrant Vector Databases

This notebook provides a comprehensive guide to understanding and using QDrant vector databases with LangChain.

## What is QDrant?

QDrant is a high-performance vector database designed for similarity search and machine learning applications. It's particularly well-suited for:

- **Semantic Search**: Find similar documents based on meaning
- **Recommendation Systems**: Suggest similar items or content
- **RAG Applications**: Retrieve relevant context for language models
- **Clustering**: Group similar data points together

## Key Concepts

### 1. Collections
A **Collection** is like a database table that stores vectors of the same dimension and type.

### 2. Points
A **Point** is a single vector with an ID and optional payload (metadata).

### 3. Distance Metrics
**Distance Metrics** determine how similarity is calculated between vectors:
- **Cosine**: Measures the angle between vectors (good for text embeddings)
- **Euclidean**: Measures straight-line distance
- **Dot Product**: Measures vector alignment

### 4. Payloads
**Payloads** are metadata attached to vectors (e.g., document text, timestamps, categories).


In [1]:
# Import required libraries
import os
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams, PointStruct
from langchain_ollama import OllamaEmbeddings
from langchain_core.documents import Document
import numpy as np

print("✅ Libraries imported successfully!")


✅ Libraries imported successfully!


In [2]:
# Initialize embedding model
embedding_model = OllamaEmbeddings(model="embeddinggemma:latest")

# Test embedding to get dimension
test_embedding = embedding_model.embed_query("test")
embedding_dim = len(test_embedding)

print(f"✅ Embedding model initialized")
print(f"Embedding dimension: {embedding_dim}")
print(f"Sample embedding (first 5 values): {test_embedding[:5]}")


✅ Embedding model initialized
Embedding dimension: 768
Sample embedding (first 5 values): [-0.21215437, 0.00081568345, 0.021561427, 0.0063646873, 0.011488259]


## 1. Setting Up QDrant

Let's start by setting up a QDrant client and creating our first collection.


In [3]:
# Create QDrant client (in-memory for this demo)
client = QdrantClient(":memory:")

# Create a collection
collection_name = "ai_knowledge_base"

try:
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(
            size=embedding_dim,  # Dimension of our embeddings
            distance=Distance.COSINE  # Distance metric for similarity
        )
    )
    print(f"✅ Collection '{collection_name}' created successfully!")
except Exception as e:
    print(f"Collection might already exist: {e}")

# Check collection info
collection_info = client.get_collection(collection_name)
print(f"Collection info: {collection_info}")


✅ Collection 'ai_knowledge_base' created successfully!
Collection info: status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> vectors_count=None indexed_vectors_count=0 points_count=0 segments_count=1 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=768, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=None, sharding_method=None, replication_factor=None, write_consistency_factor=None, read_fan_out_factor=None, on_disk_payload=None, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=None, payload_m=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=20000, flush_interval_sec=5, max_optimization_threads=1), wal_config=Wa

## 2. Working with LangChain QDrant Integration

LangChain provides a convenient wrapper around QDrant that makes it easy to use in RAG applications.


In [4]:
# Create LangChain QDrant vector store
vector_store = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embedding_model
)

print("✅ LangChain QDrant vector store created!")


✅ LangChain QDrant vector store created!


In [5]:
# Create sample documents
sample_documents = [
    Document(
        page_content="Artificial Intelligence (AI) is the simulation of human intelligence in machines.",
        metadata={"topic": "AI", "source": "intro_guide"}
    ),
    Document(
        page_content="Machine Learning is a subset of AI that enables computers to learn without explicit programming.",
        metadata={"topic": "ML", "source": "ml_basics"}
    ),
    Document(
        page_content="Deep Learning uses neural networks with multiple layers to process data.",
        metadata={"topic": "Deep Learning", "source": "neural_networks"}
    ),
    Document(
        page_content="Natural Language Processing (NLP) helps computers understand and generate human language.",
        metadata={"topic": "NLP", "source": "language_processing"}
    ),
    Document(
        page_content="Computer Vision enables machines to interpret and understand visual information.",
        metadata={"topic": "CV", "source": "visual_processing"}
    ),
    Document(
        page_content="Vector databases store high-dimensional vectors for similarity search and retrieval.",
        metadata={"topic": "Vector DB", "source": "database_guide"}
    )
]

print(f"✅ Created {len(sample_documents)} sample documents")
for i, doc in enumerate(sample_documents, 1):
    print(f"{i}. {doc.page_content[:50]}... (Topic: {doc.metadata['topic']})")


✅ Created 6 sample documents
1. Artificial Intelligence (AI) is the simulation of ... (Topic: AI)
2. Machine Learning is a subset of AI that enables co... (Topic: ML)
3. Deep Learning uses neural networks with multiple l... (Topic: Deep Learning)
4. Natural Language Processing (NLP) helps computers ... (Topic: NLP)
5. Computer Vision enables machines to interpret and ... (Topic: CV)
6. Vector databases store high-dimensional vectors fo... (Topic: Vector DB)


## 3. Adding Documents to QDrant

Now let's add our documents to the vector store and see how similarity search works.


In [6]:
# Add documents to the vector store
print("Adding documents to QDrant...")
vector_store.add_documents(sample_documents)

# Check collection stats
collection_info = client.get_collection(collection_name)
print(f"✅ Documents added successfully!")
print(f"Total points in collection: {collection_info.points_count}")
print(f"Vector dimension: {collection_info.config.params.vectors.size}")
print(f"Distance metric: {collection_info.config.params.vectors.distance}")


Adding documents to QDrant...
✅ Documents added successfully!
Total points in collection: 6
Vector dimension: 768
Distance metric: Cosine


## 4. Similarity Search with QDrant

Let's test different types of similarity searches to understand how QDrant works.


In [7]:
# Test 1: Basic similarity search
query = "What is machine learning?"
print(f"Query: '{query}'")
print("=" * 50)

# Create a retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# Search for similar documents
similar_docs = retriever.invoke(query)

print("Most similar documents:")
for i, doc in enumerate(similar_docs, 1):
    print(f"{i}. {doc.page_content}")
    print(f"   Metadata: {doc.metadata}")
    print()


Query: 'What is machine learning?'
Most similar documents:
1. Deep Learning uses neural networks with multiple layers to process data.
   Metadata: {'topic': 'Deep Learning', 'source': 'neural_networks', '_id': 'b595af36795f4c68abbd1e2a9d2b71f8', '_collection_name': 'ai_knowledge_base'}

2. Machine Learning is a subset of AI that enables computers to learn without explicit programming.
   Metadata: {'topic': 'ML', 'source': 'ml_basics', '_id': '34ebe5b1dea449fdbec23fd4172cb410', '_collection_name': 'ai_knowledge_base'}

3. Computer Vision enables machines to interpret and understand visual information.
   Metadata: {'topic': 'CV', 'source': 'visual_processing', '_id': 'af2c99a31a3c4b659dbc51e680f0c192', '_collection_name': 'ai_knowledge_base'}



In [8]:
# Test 2: Different query types
test_queries = [
    "How do neural networks work?",
    "What is computer vision?",
    "Tell me about vector databases",
    "What is natural language processing?"
]

print("Testing different query types:")
print("=" * 50)

for query in test_queries:
    print(f"\nQuery: '{query}'")
    print("-" * 30)
    
    similar_docs = retriever.invoke(query)
    for i, doc in enumerate(similar_docs[:2], 1):  # Show top 2 results
        print(f"{i}. {doc.page_content[:80]}...")
        print(f"   Topic: {doc.metadata['topic']}")
    print()


Testing different query types:

Query: 'How do neural networks work?'
------------------------------
1. Deep Learning uses neural networks with multiple layers to process data....
   Topic: Deep Learning
2. Vector databases store high-dimensional vectors for similarity search and retrie...
   Topic: Vector DB


Query: 'What is computer vision?'
------------------------------
1. Computer Vision enables machines to interpret and understand visual information....
   Topic: CV
2. Vector databases store high-dimensional vectors for similarity search and retrie...
   Topic: Vector DB


Query: 'Tell me about vector databases'
------------------------------
1. Vector databases store high-dimensional vectors for similarity search and retrie...
   Topic: Vector DB
2. Computer Vision enables machines to interpret and understand visual information....
   Topic: CV


Query: 'What is natural language processing?'
------------------------------
1. Vector databases store high-dimensional vectors for s

## 5. Advanced QDrant Features

Let's explore some advanced features like filtering and custom similarity thresholds.


In [10]:
# Test 3: Filtering by metadata
print("Testing metadata filtering:")
print("=" * 50)

# Search only for AI-related documents
ai_retriever = vector_store.as_retriever(
    search_kwargs={
        "k": 3,
        "filter": {"must": [{"key": "topic", "match": {"value": "AI"}}]}
    }
)

query = "What is artificial intelligence?"
ai_docs = ai_retriever.invoke(query)

print(f"Query: '{query}'")
print("AI-filtered results:")
for i, doc in enumerate(ai_docs, 1):
    print(f"{i}. {doc.page_content}")
    print(f"   Topic: {doc.metadata['topic']}")
    print()


Testing metadata filtering:
Query: 'What is artificial intelligence?'
AI-filtered results:


In [11]:
# Test 4: Similarity scores
print("Testing similarity scores:")
print("=" * 50)

# Get similarity scores
query_embedding = embedding_model.embed_query("machine learning algorithms")

# Use the client directly to get scores
search_result = client.search(
    collection_name=collection_name,
    query_vector=query_embedding,
    limit=3,
    with_payload=True
)

print(f"Query: 'machine learning algorithms'")
print("Results with similarity scores:")
for i, result in enumerate(search_result, 1):
    print(f"{i}. Score: {result.score:.4f}")
    print(f"   Content: {result.payload['page_content']}")
    print(f"   Topic: {result.payload['metadata']['topic']}")
    print()


Testing similarity scores:
Query: 'machine learning algorithms'
Results with similarity scores:
1. Score: 0.8138
   Content: Deep Learning uses neural networks with multiple layers to process data.
   Topic: Deep Learning

2. Score: 0.7918
   Content: Vector databases store high-dimensional vectors for similarity search and retrieval.
   Topic: Vector DB

3. Score: 0.7810
   Content: Computer Vision enables machines to interpret and understand visual information.
   Topic: CV



/var/folders/2n/cyq8dv9j683_ynffwh1tw3_c0000gn/T/ipykernel_88574/1172007802.py:9: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  search_result = client.search(


## 6. Building a RAG System with QDrant

Now let's put it all together and build a complete RAG system using QDrant.


In [12]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Initialize chat model
chat_model = ChatOllama(model="gpt-oss:20b", temperature=0.7)

# Create RAG prompt template
rag_prompt = ChatPromptTemplate.from_template("""
Context: {context}

Question: {question}

Answer the question based on the provided context. If the context doesn't contain enough information, say so.
""")

# Create RAG chain
rag_chain = rag_prompt | chat_model | StrOutputParser()

def create_rag_system(vector_store):
    """Create a complete RAG system using QDrant."""
    retriever = vector_store.as_retriever(search_kwargs={"k": 3})
    
    def rag_query(question):
        # Retrieve relevant documents
        docs = retriever.invoke(question)
        context = "\\n".join([doc.page_content for doc in docs])
        
        # Generate response
        response = rag_chain.invoke({"context": context, "question": question})
        return response, docs
    
    return rag_query

# Create RAG system
rag_system = create_rag_system(vector_store)

print("✅ RAG system created with QDrant!")


✅ RAG system created with QDrant!


In [13]:
# Test the RAG system
test_questions = [
    "What is machine learning?",
    "How does deep learning work?",
    "What is the difference between AI and ML?",
    "What are vector databases used for?"
]

print("Testing RAG system:")
print("=" * 50)

for question in test_questions:
    print(f"\\nQuestion: {question}")
    print("-" * 40)
    
    response, retrieved_docs = rag_system(question)
    
    print(f"Answer: {response}")
    print(f"\\nRetrieved {len(retrieved_docs)} documents:")
    for i, doc in enumerate(retrieved_docs, 1):
        print(f"  {i}. {doc.page_content[:60]}... (Topic: {doc.metadata['topic']})")
    print("\\n" + "="*50)


Testing RAG system:
\nQuestion: What is machine learning?
----------------------------------------
Answer: Machine learning is a subset of artificial intelligence that enables computers to learn from data and improve their performance on a task without being explicitly programmed for every specific case.
\nRetrieved 3 documents:
  1. Deep Learning uses neural networks with multiple layers to p... (Topic: Deep Learning)
  2. Machine Learning is a subset of AI that enables computers to... (Topic: ML)
  3. Computer Vision enables machines to interpret and understand... (Topic: CV)
\n==================================================
\nQuestion: How does deep learning work?
----------------------------------------
Answer: Deep learning works by using neural networks that consist of many interconnected layers. These layers transform input data step by step, allowing the network to learn complex patterns and representations automatically from the data.
\nRetrieved 3 documents:
  1. Vector da